# Vet Clinics in Berlin – OSM + LOR Spatial Join (v0)

This notebook takes the vet clinics snapshot extracted from OpenStreetMap
via OSMNX (`00_vet_clinics_osmnx_download.ipynb`) and enriches it with
Berlin district and neighborhood information using the LOR / Ortsteile
polygons.

The output is an intermediate **v0 dataset** that:

- Contains OSM vet clinic attributes (name, address, contact, opening hours).
- Has geographic coordinates (`lat`, `lon`) and point geometries.
- Is spatially joined to Berlin LOR polygons.
- Includes district and neighborhood names (`district`, `neighborhood`) and IDs
  (`district_id`, `neighborhood_id`) following the existing mapping example.

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)

## 1. Load OSM vet clinics snapshot (OSMNX)

We use the GeoJSON snapshot generated by
`00_vet_clinics_osmnx_download.ipynb`:

- `sources/osmnx_berlin_vet_clinics_latest.geojson`

This file is expected to contain:
- `source_osm_id`
- OSM address and contact tags (`addr:*`, `phone`, `contact:phone`, etc.)
- `opening_hours`, `operator`, `emergency`, `wheelchair`, etc.
- `lat`, `lon`
- point `geometry`

In [2]:
# Working directory is expected to be:
# .../layered-populate-data-pool-da/veterinary_clinics

osm_geojson_path = Path("sources/osmnx_berlin_vet_clinics_latest.geojson")

gdf_vets = gpd.read_file(osm_geojson_path)

display(gdf_vets.head())
print(f"Number of OSM vet clinic features (OSMNX snapshot): {len(gdf_vets)}")
print("CRS:", gdf_vets.crs)

,element,id,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,email,contact:email,website,contact:website,opening_hours,operator,wheelchair,wheelchair:description,emergency,lat,lon,geometry
0,node,268917040,"('node', 268917040)",Tierarztpraxis am Urban,Baerwaldstraße,69,10961,Berlin,None,None,None,None,None,None,"Mo-Sa 10:00-12:00, Mo 17:00-19:00, Tu,We,Fr 16...",None,no,None,None,52.495684,13.405233,POINT (13.40523 52.49568)
1,node,299795048,"('node', 299795048)",Dr. med. vet. Elke Hartwig,Straße 48,67,13125,Berlin,+49 30 9437820,None,None,None,http://www.tierarztpraxis-hartwig.de/,None,"Mo,Tu,Th,Fr 10:00-12:00, Mo-Fr 15:00-18:00",None,limited,None,None,52.606286,13.479555,POINT (13.47955 52.60629)
2,node,347294456,"('node', 347294456)",Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207,Berlin,+49 30 7738321,None,None,None,https://www.tierarztpraxis-soerensen.de/,None,"Mo-Fr 09:00-20:00; Sa, Su 10:00-18:00",None,yes,None,None,52.429722,13.320133,POINT (13.32013 52.42972)
3,node,394867279,"('node', 394867279)",Tierarztpraxis Jeanette Koepsel,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,52.535199,13.270573,POINT (13.27057 52.5352)
4,node,411550894,"('node', 411550894)",Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621,Berlin,+49 30 53018585,None,info@tierarzt-kaulsdorf.de,None,https://www.tierarzt-kaulsdorf.de/,None,"Mo-Fr 09:00-19:00 open ""tel. Terminvereinbarun...",Dr. Berit Miels;Dr. Mathias Kochert,None,None,None,52.509511,13.589635,POINT (13.58964 52.50951)


Number of OSM vet clinic features (OSMNX snapshot): 175
CRS: EPSG:4326


## 2. Load Berlin LOR / Ortsteile polygons

We use the LOR / Ortsteile GeoJSON as the reference layer for
districts and neighborhoods:

- `sources/raw_berlin_lor_ortsteile.geojson`

From this dataset we will keep:

- `gml_id`         → LOR polygon identifier (`lor_id`)
- `BEZIRK`         → district name (`district`)
- `OTEIL`          → neighborhood name (`neighborhood`)
- `spatial_name`   → neighborhood ID (`neighborhood_id`)
- `geometry`       → polygons for spatial join

In [3]:
lor_path = Path("sources/raw_berlin_lor_ortsteile.geojson")
gdf_lor = gpd.read_file(lor_path)

display(gdf_lor.head())
print("LOR CRS:", gdf_lor.crs)
print("LOR columns:", list(gdf_lor.columns))

,gml_id,spatial_name,spatial_alias,spatial_type,OTEIL,BEZIRK,FLAECHE_HA,geometry
0,re_ortsteil.0101,0101,Mitte,Polygon,Mitte,Mitte,1063.8748,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,0102,Moabit,Polygon,Moabit,Mitte,768.7909,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,0103,Hansaviertel,Polygon,Hansaviertel,Mitte,52.5337,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,0104,Tiergarten,Polygon,Tiergarten,Mitte,516.0672,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,0105,Wedding,Polygon,Wedding,Mitte,919.9112,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


LOR CRS: EPSG:4326
LOR columns: ['gml_id', 'spatial_name', 'spatial_alias', 'spatial_type', 'OTEIL', 'BEZIRK', 'FLAECHE_HA', 'geometry']


## 3. Subset LOR columns needed for the join

We keep only:

- `gml_id`        → `lor_id`
- `BEZIRK`        → `district`
- `OTEIL`         → `neighborhood`
- `spatial_name`  → `neighborhood_id`
- `geometry`      → polygons used for the spatial join

In [4]:
lor_id_col = "gml_id"            # LOR polygon ID
district_col = "BEZIRK"          # district name
neighborhood_col = "OTEIL"       # neighborhood name
neighborhood_id_col = "spatial_name"  # neighborhood ID (used in mapping example)

cols_lor_keep = [
    lor_id_col,
    district_col,
    neighborhood_col,
    neighborhood_id_col,
    "geometry",
]

gdf_lor_subset = gdf_lor[cols_lor_keep].copy()

display(gdf_lor_subset.head())

,gml_id,BEZIRK,OTEIL,spatial_name,geometry
0,re_ortsteil.0101,Mitte,Mitte,0101,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,re_ortsteil.0102,Mitte,Moabit,0102,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,re_ortsteil.0103,Mitte,Hansaviertel,0103,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,re_ortsteil.0104,Mitte,Tiergarten,0104,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,re_ortsteil.0105,Mitte,Wedding,0105,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


## 4. Ensure both layers share a common CRS

Before spatial joining, we confirm that both:

- vet clinics (`gdf_vets`)
- LOR polygons (`gdf_lor_subset`)

use a compatible CRS. If needed, we reproject LOR to match the vet clinics CRS.

In [5]:
print("Vets CRS:", gdf_vets.crs)
print("LOR subset CRS:", gdf_lor_subset.crs)

# Reproject LOR to the vets CRS if they differ
if gdf_lor_subset.crs != gdf_vets.crs:
    gdf_lor_subset = gdf_lor_subset.to_crs(gdf_vets.crs)
    print("Reprojected LOR subset to:", gdf_lor_subset.crs)

Vets CRS: EPSG:4326
LOR subset CRS: EPSG:4326


## 5. Spatial join: assign district and neighborhood to each vet clinic

We perform a spatial join where each vet clinic point is matched to the
LOR polygon that contains it, using `predicate="within"`.

The result adds:

- `lor_id`          (from `gml_id`)
- `district`        (from `BEZIRK`)
- `neighborhood`    (from `OTEIL`)
- `neighborhood_id` (from `spatial_name`)

In [6]:
# Spatial join: vet clinic point within LOR polygon
gdf_vets_lor = gpd.sjoin(
    gdf_vets,
    gdf_lor_subset,
    how="left",
    predicate="within",
)

# Drop join index column if present
if "index_right" in gdf_vets_lor.columns:
    gdf_vets_lor = gdf_vets_lor.drop(columns=["index_right"])

# Rename LOR columns to match mapping example
rename_map = {
    lor_id_col: "lor_id",
    district_col: "district",
    neighborhood_col: "neighborhood",
    neighborhood_id_col: "neighborhood_id",
}

gdf_vets_lor = gdf_vets_lor.rename(columns=rename_map)

# Ensure lat/lon are present (they should come from the OSMNX snapshot)
if "lon" not in gdf_vets_lor.columns:
    gdf_vets_lor["lon"] = gdf_vets_lor.geometry.x
if "lat" not in gdf_vets_lor.columns:
    gdf_vets_lor["lat"] = gdf_vets_lor.geometry.y

gdf_vets_lor[
    ["source_osm_id", "name", "district", "neighborhood", "neighborhood_id", "lat", "lon"]
].head(10)

,source_osm_id,name,district,neighborhood,neighborhood_id,lat,lon
0,"('node', 268917040)",Tierarztpraxis am Urban,Friedrichshain-Kreuzberg,Kreuzberg,0202,52.495684,13.405233
1,"('node', 299795048)",Dr. med. vet. Elke Hartwig,Pankow,Karow,0305,52.606286,13.479555
2,"('node', 347294456)",Tierarztpraxis Dr. Bernhard Sörensen,Steglitz-Zehlendorf,Lichterfelde,0602,52.429722,13.320133
3,"('node', 394867279)",Tierarztpraxis Jeanette Koepsel,Spandau,Siemensstadt,0503,52.535199,13.270573
4,"('node', 411550894)",Kleintierarztpraxis Berlin Kaulsdorf,Marzahn-Hellersdorf,Kaulsdorf,1003,52.509511,13.589635
5,"('node', 581887970)",Eva Klein,Steglitz-Zehlendorf,Nikolassee,0606,52.428817,13.193529
6,"('node', 603392630)",Klein- und Heimtierklinik,Steglitz-Zehlendorf,Zehlendorf,0604,52.430298,13.237360
7,"('node', 702363921)",Tierarztpraxis im Frauenviertel - Firchow,Neukölln,Rudow,0804,52.404260,13.507166
8,"('node', 703135335)",Kleintierpraxis Kladow,Spandau,Kladow,0506,52.460888,13.121554
9,"('node', 978560965)",Carsten Schiller,Treptow-Köpenick,Niederschöneweide,0905,52.453108,13.524567


## 6. Map district names to district_id

We now map the district names to official `district_id` values following
the existing mapping example used in the repository. This ensures that
the vet clinics layer is consistent with other Berlin-based layers.

In [7]:
# District mapping (official codes as strings) – aligned with the mapping example
district_mapping = {
    "Mitte": "11001001",
    "Friedrichshain-Kreuzberg": "11002002",
    "Pankow": "11003003",
    "Charlottenburg-Wilmersdorf": "11004004",
    "Spandau": "11005005",
    "Steglitz-Zehlendorf": "11006006",
    "Tempelhof-Schöneberg": "11007007",
    "Neukölln": "11008008",
    "Treptow-Köpenick": "11009009",
    "Marzahn-Hellersdorf": "11010010",
    "Lichtenberg": "11011011",
    "Reinickendorf": "11012012",
}

# Map district name → district_id
gdf_vets_lor["district_id"] = (
    gdf_vets_lor["district"]
    .map(district_mapping)
    .astype("string")
)

# Quick sanity check of unique combinations
gdf_vets_lor[
    ["district", "district_id", "neighborhood", "neighborhood_id"]
].drop_duplicates().head(15)

,district,district_id,neighborhood,neighborhood_id
0,Friedrichshain-Kreuzberg,11002002,Kreuzberg,0202
1,Pankow,11003003,Karow,0305
2,Steglitz-Zehlendorf,11006006,Lichterfelde,0602
3,Spandau,11005005,Siemensstadt,0503
4,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003
5,Steglitz-Zehlendorf,11006006,Nikolassee,0606
6,Steglitz-Zehlendorf,11006006,Zehlendorf,0604
7,Neukölln,11008008,Rudow,0804
8,Spandau,11005005,Kladow,0506
9,Treptow-Köpenick,11009009,Niederschöneweide,0905


## 7. Prepare v0 dataset and export to `cache/`

We now build the v0 dataset combining:

- OSM attributes (name, address, contact, opening_hours, operator, emergency).
- Coordinates (`lat`, `lon`).
- LOR-based context:
  - `lor_id`
  - `district`, `district_id`
  - `neighborhood`, `neighborhood_id`.

This v0 file is the input to the cleaning and normalization step in
`02_vet_clinics_cleaning_and_normalization.ipynb`.

In [8]:
from pathlib import Path

# Define columns to keep in v0
cols_v0 = [
    "source_osm_id",
    "name",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "phone",
    "contact:phone",
    "email",
    "contact:email",
    "website",
    "contact:website",
    "opening_hours",
    "operator",
    "wheelchair",
    "wheelchair:description",
    "emergency",
    "lat",
    "lon",
    "geometry",          # <-- keep geometry in v0
    "lor_id",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
]

# Keep only columns that actually exist
cols_v0 = [c for c in cols_v0 if c in gdf_vets_lor.columns]

df_v0 = gdf_vets_lor[cols_v0].copy()


output_v0_csv = Path("cache/vets_osm_berlin_with_lor_latest_v0.csv")

# Ensure cache directory exists
output_v0_csv.parent.mkdir(parents=True, exist_ok=True)

df_v0.to_csv(output_v0_csv, index=False)

print("Exported v0 file to:")
print(" -", output_v0_csv)
print("Shape:", df_v0.shape)

Exported v0 file to:
 - cache/vets_osm_berlin_with_lor_latest_v0.csv
Shape: (175, 25)
